# 07 — Trajectory ingestion: the eps_nuc pin and the stall

The node that made the shipped test trajectories safe to consume
(`gnn_nucleo/data/trajectories.py`). Two things had to be settled before any
trajectory-based measurement could be trusted:

1. **The eps_nuc convention** — trajectory files disagree with the training
   CSVs. Pinned (Step-6 Task 0): trajectory `eps_nuc` is **INTEGRATED** per
   output row [erg/g], **net of neutrino losses**, with **no 1e16
   normalization**; `eps_neu` is a **rate** [erg/g/s].
2. **The stall** — composition freezes mid-file. Step 5 read this as
   trajectory death; Step 6 reinterpreted it as *arrival* at the
   (bug-displaced) attractor, with weak-sector evolution continuing after.
   Two guard semantics now exist and they disagree on 459/1508 shipped files.

Exploratory only — citable values are the RESULTS.md 2026-07-11 rows.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd()
if not (_here / "nbsupport.py").exists():
    _root = next(p for p in [_here, *_here.parents] if (p / "pyproject.toml").exists())
    _here = _root / "notebooks" / "phase0"
sys.path.insert(0, str(_here))

import matplotlib.pyplot as plt
import numpy as np

import nbsupport as nbs
from gnn_nucleo.data.trajectories import (  # cheap import (numpy); graph import is lazy
    STALL_TOL,
    eps_nuc_integrated,
    eps_nuc_rate,
    load_trajectory,
    select_rows,
    stall_row,
    zenodo_trajectory_dir,
)

nbs.style()
QUICK = nbs.QUICK

In [ ]:
nbs.provenance_header(
    "07",
    "Trajectory ingestion — eps_nuc pin + stall semantics",
    nbs.status_of([11]),
    results_rows=[
        "2026-07-11: eps_nuc convention PIN — integrated per row, net of ν losses, no 1e16 normalization; sign agreement 0.993/0.998, log-ratio IQR ≈ 0, slope vs dt-decade 0.000 across 20 decades (rate reading shows the wrong-convention slope +1.00)",
        "2026-07-11: stall-rule regression — terminal vs first_quiet differ on 459/1508 shipped files; 'stall' = ARRIVAL at the Appendix-B bug-displaced attractor, not trajectory death",
        "2026-07-10: trajectory-anomaly row — composition frozen (max |ΔX| < 1e-10) from median age 2.2e5 s / 3.9e4 s (mesa_80/151); frozen states are NOT NSE",
    ],
    data=["data/zenodo/.../test_datasets/mesa_80/mesa_80_output_files/*.txt (READ-ONLY)"],
    scripts=[
        "scripts/step6_eps_pin.py",
        "scripts/step5_qse.py",
        "tests/test_trajectories.py (stall-rule regression)",
    ],
)

## Load: shipped trajectories, default row selection

`select_rows(traj, prestall=True, mode="terminal")` — **these defaults apply
to every figure below**. `prestall=True` drops the trajectory-ending quiet
tail; post-stall rows require an explicit override.

In [ ]:
NET = "mesa_80"
files = sorted(zenodo_trajectory_dir(NET).glob("output_T_*.txt"))
n_scan = 40 if QUICK else len(files)
print(f"{len(files)} shipped trajectories for {NET}; scanning {n_scan}")

scan = []
for f in files[:n_scan]:
    tr = load_trajectory(NET, f)
    term, first = stall_row(tr.X, mode="terminal"), stall_row(tr.X, mode="first_quiet")
    scan.append((f, tr, term, first))

differ = [s for s in scan if s[2] != s[3]]
print(f"rules disagree on {len(differ)}/{len(scan)} scanned files "
      f"(measured over all shipped files: 459/1508 — RESULTS.md 2026-07-11)")

## Figure 1 — the eps_nuc convention pin

The discriminator: read the file's `eps_nuc` column BOTH ways (as an
integrated energy per row, and as a rate to be multiplied by the row's dt)
and compare each against an independent composition-route energy. The wrong
reading acquires a **slope of +1 against the dt decade** — it scales with the
step size, which a physical energy per row must not do. The correct reading
is flat.

In [ ]:
tr = scan[0][1]
rows = select_rows(tr)[1:]  # default guard; drop row 0 (no preceding interval)
dt = tr.dt[rows]
integ = eps_nuc_integrated(tr)[rows]  # erg/g   (the PIN)
rate = eps_nuc_rate(tr)[rows]  # erg/g/s (the alternative reading)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
ax = axes[0]
ax.loglog(dt, np.abs(integ), ".", ms=4, label="read as INTEGRATED [erg/g] — the pin", color="#009E73")
ax.loglog(dt, np.abs(rate), ".", ms=4, label="read as RATE × dt [erg/g]", color="#D55E00", alpha=0.6)
ax.set_xlabel("row dt [s]")
ax.set_ylabel("|energy per row| [erg/g]")
ax.set_title(f"{NET} · logT={tr.logT} logRho={tr.logRho}")
ax.legend(fontsize=8)

ax = axes[1]
ok = (dt > 0) & np.isfinite(integ) & (np.abs(integ) > 0)
ld = np.log10(dt[ok])
for vals, label, c in [(integ[ok], "INTEGRATED (pin)", "#009E73"), (rate[ok], "RATE reading", "#D55E00")]:
    lv = np.log10(np.abs(vals))
    sl = np.polyfit(ld, lv, 1)[0]
    ax.plot(ld, lv - np.median(lv), ".", ms=3, alpha=0.5, color=c, label=f"{label}: slope {sl:+.2f}")
ax.set_xlabel("log10 row dt [s]")
ax.set_ylabel("log10 |energy|, centred")
ax.set_title("Slope vs dt decade — the convention discriminator")
ax.legend(fontsize=8)
nbs.caption(
    fig,
    "Quick-look on one trajectory of the pinned convention: the integrated reading is "
    "dt-independent while the rate reading tracks dt. The measurement proper (sign agreement "
    "0.993/0.998, log-ratio IQR ≈ 0, slope 0.000 for the pin vs +1.00 for the wrong reading, "
    "across 20 dt decades; matched to bbq source lib_bbq.f90:453) is scripts/step6_eps_pin.py. "
    "Training CSVs use a DIFFERENT convention (÷1e16) — never mix them.",
    results=["RESULTS.md 2026-07-11 eps_nuc pin rows (scripts/step6_eps_pin.py, commit efaffd4)"],
    scripts=["scripts/step6_eps_pin.py"],
)

## Figure 2 — stall anatomy: `terminal` vs `first_quiet`

`max|ΔX|` per interval against age. `first_quiet` (the Step-5 rule) cuts at
the FIRST interval below `STALL_TOL` — which is *arrival at the attractor*.
`terminal` (the current default) cuts only the trajectory-ending quiet tail,
keeping the post-arrival rows where the **weak sector is still evolving**.

In [ ]:
pick = differ[0] if differ else scan[0]
f, tr2, term, first = pick
dX = np.abs(np.diff(tr2.X, axis=0)).max(axis=1)
age = tr2.age[1:]

fig, ax = plt.subplots(figsize=(10, 4.6))
ax.loglog(np.maximum(age, 1e-10), np.maximum(dX, 1e-18), lw=1, color="#0072B2")
ax.axhline(STALL_TOL, color="#333333", ls="--", lw=1, label=f"STALL_TOL = {STALL_TOL:g}")
for row, label, c in [(first, "first_quiet cut (Step-5 rule)", "#D55E00"),
                      (term, "terminal cut (current default)", "#009E73")]:
    if 0 < row < len(age):
        ax.axvline(max(age[row - 1], 1e-10), color=c, ls="-", lw=1.6, label=label)
ax.set_xlabel("age [s]")
ax.set_ylabel("max |ΔX| per interval")
ax.set_title(f"Stall anatomy — {f.name} (logT={tr2.logT}, logRho={tr2.logRho})")
ax.legend(fontsize=8, loc="lower left")
print(f"{f.name}: first_quiet row {first}, terminal row {term}, of {tr2.n_rows} rows")
nbs.caption(
    fig,
    "The strong sector goes quiet early (first_quiet) but the file continues — under the Step-5 "
    "rule everything to the right was discarded as 'stalled'. Step 6 established those rows are "
    "LEGITIMATE relaxed label-dynamics states (arrival at the bug-displaced attractor, notebook "
    "09), so the kill-test manifold uses terminal-tail selection and keeps them.",
    results=[
        "RESULTS.md 2026-07-11 stall-rule regression rows (459/1508 files; tests/test_trajectories.py, commit ea970b4)",
        "RESULTS.md 2026-07-10 trajectory-anomaly row (frozen states are NOT NSE)",
    ],
    scripts=["scripts/step5_qse.py", "tests/test_trajectories.py"],
)

## Figure 3 — what happens after arrival: the weak sector keeps going

This is why the reinterpretation matters. Past the first-quiet row the
strong sector is frozen at the displaced attractor, but Yₑ continues to
evolve through weak reactions — the very signal Target A must predict
(invariant #3: a design that zeroes dYₑ to make drift vanish is wrong).

In [ ]:
from gnn_nucleo.graph.isotopes import load_isotope_table  # lazy: pulls pynucastro once

tab = load_isotope_table(NET)
Z = np.asarray(tab.Z, dtype=np.float64)
A = np.asarray(tab.A, dtype=np.float64)
ye_t = (tr2.X * (Z / A)[None, :]).sum(axis=1)  # Yₑ = Σ Zᵢ Xᵢ / Aᵢ

fig, ax = plt.subplots(figsize=(10, 4.6))
ax.semilogx(np.maximum(tr2.age, 1e-10), ye_t, lw=1.5, color="#0072B2")
if 0 < first < tr2.n_rows:
    ax.axvline(max(tr2.age[first], 1e-10), color="#D55E00", lw=1.6,
               label="first_quiet (strong sector arrives)")
    post = slice(first, None)
    ax.fill_between(np.maximum(tr2.age[post], 1e-10), ye_t[post].min(), ye_t[post].max(),
                    color="#E69F00", alpha=0.2, label="post-arrival: weak sector still evolving")
    print(f"Yₑ drift after arrival: {ye_t[first]:.6f} → {ye_t[-1]:.6f} "
          f"(Δ = {ye_t[-1] - ye_t[first]:+.2e}) over {tr2.age[-1] - tr2.age[first]:.3g} s")
ax.set_xlabel("age [s]")
ax.set_ylabel("Yₑ = Σ Zᵢ Xᵢ / Aᵢ")
ax.set_title(f"Weak-driven Yₑ evolution continues past the 'stall' — {f.name}")
ax.legend(fontsize=8)
nbs.caption(
    fig,
    "Composition 'freezing' is a STRONG-sector statement only: Yₑ keeps drifting under weak "
    "reactions after arrival, which is why post-arrival rows are legitimate relaxed states and "
    "why the pre-stall-only restriction was superseded for the kill-test (checklist rows 6, 8, 9). "
    "Quick-look on one trajectory; the census is in the cited rows.",
    results=[
        "RESULTS.md 2026-07-11 stall-reinterpretation rows",
        "docs/phase0-checklist.md 2026-07-12 update (rows 6/8/9 measured on the relaxed manifold)",
    ],
    scripts=["scripts/run_killtest.py --relaxed --include-reruns"],
)

## TODO (stub)

- **Full 459/1508 rule-difference census** across both networks — measured
  (RESULTS.md 2026-07-11); the regression lives in
  `tests/test_trajectories.py`. This notebook scans 40 files in QUICK mode.

## What this notebook does NOT show

- Why the attractor is displaced (the gh-575 label pathology): notebook 09.
- The bbq reruns that provide non-stalling relaxed trajectories: notebook 08.
- Training-CSV eps convention (÷1e16): notebook 01's stub /
  `scripts/check_training_csvs.py`.